# A reproducible learning problem and a leakage-safe data workflow

**Core time:** 90–120 minutes. **Prerequisites:** setup guide, arrays and basic probability. **Outcomes:** define inputs/targets and the deployment question; reserve an appropriate test set; fit preprocessing only on training data; preserve split identifiers and a baseline.

## 1. Define the prediction event

A supervised-learning example contains an input available at prediction time and a target representing the desired output. Before fitting anything, describe what generates an observation, when a prediction is made and which information exists at that time. A feature measured after the outcome can predict well in a retrospective table while being unusable in practice. Duplicate records can put nearly identical examples in both training and evaluation sets.

Our teaching task classifies 8-by-8 handwritten digits from scikit-learn's bundled dataset. The inputs are 64 pixel intensities; the target is one of ten digit classes. This is a small classification benchmark, not evidence about a deployed handwriting system. Treat the data description and original attribution as part of the artifact. Scaling intensities by the documented constant 16 uses a known measurement range; estimating a mean or standard deviation from data would instead be a fitting step.

## 2. Three sets have three jobs

The training set estimates model parameters and preprocessing statistics. The validation set supports hyperparameter selection, early stopping and threshold choices. The test set estimates final performance after those choices are frozen. Cross-validation can replace a single validation split, but every preprocessing step must be refitted inside each training fold. Reusing test scores to choose another model turns the test set into a development set.

For independent examples, a seeded stratified split can preserve approximate class proportions. For grouped observations, separate groups rather than rows. For future forecasting, preserve temporal direction and consider an appropriate gap. Random splitting is not automatically the most rigorous choice; the correct split matches the intended generalization question.



In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
data = load_digits()
idx = np.arange(len(data.target))
train, rest = train_test_split(idx, test_size=0.4,
    stratify=data.target, random_state=7)
validation, test = train_test_split(rest, test_size=0.5,
    stratify=data.target[rest], random_state=8)
assert not (set(train)&set(validation) or set(train)&set(test)
            or set(validation)&set(test))
assert len(train)+len(validation)+len(test) == len(idx)
print(len(train), len(validation), len(test))



Saving integer indices is stronger than recording only the seed: it preserves the actual split even if another library version changes sampling behavior. Record dataset version, ordering and a hash too, because indices have meaning only relative to a specific ordered dataset.

## 3. Leakage can enter before the model exists

Imputation, standardization, feature selection and PCA learn something from data. Fit them on training and apply the fitted transformation to validation/test. A pipeline combines preprocessing and the estimator into one fitted object, making it easier to repeat the procedure correctly during cross-validation. It does not protect against every leakage source: an answer-containing feature or an incorrect group split still defines the wrong experiment.



In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X = data.data.astype(float)
y = data.target
model = make_pipeline(StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=7))
model.fit(X[train], y[train])
print("Validation accuracy:", model.score(X[validation], y[validation]))
# Reserve test indices until model selection is frozen.



Both validation and future inputs use the scaler fitted on training observations. Centering validation data by its own mean is a different transformation and can use information about the evaluation population that would not be available under the intended prediction protocol.

## 4. Worked example: a seemingly harmless mean

Training values are `[0,2]` and a held-out value is 100. The training mean is 1, so centered training values are `[-1,1]`. Including the held-out value makes the mean 34 and centered training values `[-34,-32]`. The preprocessing now depends on the held-out population before training begins. This exaggerated example makes the dependency visible; leakage can also be subtle and can create optimistic or otherwise distorted estimates.

## 5. Establish a baseline and an audit trail

A majority-class classifier supplies a minimal sanity baseline. A linear model tests whether a sophisticated architecture is needed. Record class counts, missing values, duplicate/group checks, preprocessing and metric definitions. The baseline should use the same legal information and evaluation split as the proposed model. Deliberately weakening it to inflate an improvement is not a controlled comparison.

Run `python lab02_baselines.py --output outputs/baselines`. Inspect `split_indices.json`, the environment record and validation-selection table. The lab chooses regularization using validation and records test metrics after selection. Adding candidates after reading the test score would change that protocol. Preserve unsuccessful candidates; selection evidence should not consist only of the winner.

## Exercises and self-check

1. Design a split for photographs from 50 people when deployment concerns new people. **Check:** group by person, keeping each person's photographs in one partition; class stratification may need extra care.
2. Why does fitting a scaler before cross-validation leak information? **Check:** each validation fold influences the transform applied to its corresponding training fold.
3. Create a manifest containing data source, license/usage terms, prediction time, unit of analysis, split hashes and label definition. Identify one plausible shortcut feature and how you would test for it.
4. A student reruns random splits until accuracy is high. Explain what was selected and how to repair the report. **Check:** the evaluation partition became a tuned choice; disclose it and reserve a genuinely independent final assessment.

## Primary reading

- [scikit-learn: common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html): audit preprocessing dependencies using the leakage examples.
- [scikit-learn: `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html): inspect data shape, intensity range and attribution.
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html): compare independent, grouped and time-ordered evaluation.


# Supervised baselines, losses and regularization

**Core time:** 120 minutes. **Prerequisites:** likelihood, gradients and data splitting. **Outcomes:** relate losses to modeling assumptions; derive logistic gradients; select regularization with validation data; interpret a baseline beyond headline accuracy.

## 1. A model family encodes a restriction

Linear regression predicts a real-valued response with an affine function. Logistic regression applies a sigmoid to an affine score to model a binary class probability. Despite its name, it is a classification model. A multiclass linear classifier produces one logit per class and uses softmax. These models cannot represent every nonlinear boundary in raw inputs; feature engineering changes the space in which the decision is linear.

A supervised objective averages a per-example loss and may add regularization. Squared error emphasizes large residuals and corresponds to a Gaussian-noise likelihood under particular assumptions. Cross-entropy corresponds to a categorical/Bernoulli conditional likelihood. The loss is a surrogate for the eventual decision metric; a lower loss need not change predicted labels or improve every thresholded metric.

## 2. Derive binary logistic regression

For a scalar score z, sigmoid produces probability p. Binary cross-entropy penalizes assigning low probability to the observed label. Combining its derivative with the sigmoid derivative simplifies the derivative with respect to the logit to `p-y`.

```math
p_i=\sigma(x_i^\top w+b),\quad
L=-\frac1n\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)],\quad
\nabla_wL=\frac1nX^\top(p-y).
```

For `x=[1,2]`, `w=[0,0]`, `b=0` and `y=1`, p is 0.5 and the one-example weight gradient is `[-0.5,-1]`. A descent step increases both weights, increasing the logit and moving toward the positive label. If y were zero, the gradient reverses. This sign check catches an implementation error before a large training run.

## 3. Stable losses work directly with logits

Extreme logits can make an explicitly computed sigmoid round to one or zero. A stable binary loss uses `logaddexp(0,z)-y*z` instead of taking logs of rounded probabilities. For multiclass problems, use stable log-sum-exp. Framework functions designed for logits implement these calculations; inspect the target type and reduction convention.



In [ ]:
import numpy as np
X = np.array([[1.,2.],[-1.,-2.]])
y = np.array([1.,0.])
w = np.zeros(2)
for _ in range(100):
    logits = X@w
    p = 1/(1+np.exp(-np.clip(logits,-50,50)))
    gradient = X.T@(p-y)/len(y) + 0.1*w
    w -= 0.1*gradient
loss = np.mean(np.logaddexp(0,X@w)-y*(X@w)) + 0.05*(w@w)
assert loss < np.log(2)
print(w,loss)



This tiny separable dataset verifies update direction and numerical behavior, not generalization. The L2 term prevents weights from growing indefinitely merely to make perfectly separated training probabilities more extreme. Clipping in the educational sigmoid protects the exponential; a production implementation should use a numerically stable library operation.

## 4. Regularization changes the preferred solution

L2 discourages large coefficients. L1 can produce exact zeros under suitable optimization. Neither penalty is automatically fair across differently scaled features: standardization changes the meaning of coefficient magnitude. Put scaling inside the fitted pipeline and document it. In scikit-learn logistic regression, C is inverse regularization strength; smaller C generally means stronger regularization. It is not the lambda multiplying a penalty in the derivation above.

A learning curve compares training and validation behavior as data quantity or optimization changes. Poor training quality might reflect insufficient expressiveness, excessive regularization, unsuccessful optimization or a bug. Strong training and weak validation quality might reflect overfitting or distribution mismatch. These patterns motivate diagnoses; they do not uniquely identify the cause.

## 5. A controlled selection protocol

Predeclare a grid such as `C in [0.01,0.1,1,10]`. Fit candidates on training data, compare the same validation metric and use a stated tie rule. Freeze the winner and evaluate test once. Record all validation candidates, including weak ones. If retraining on train plus validation, explain that additional step and keep test untouched; the teaching baseline keeps the original training fit to make the selection chain transparent.

Run `lab02_baselines.py`. Compare the majority baseline with the selected linear model and inspect classwise recall and the confusion matrix. Scores are measurements to be produced by your run, not targets to copy from a lesson. A method that fails to beat the linear baseline can still teach you that a more complex architecture was unnecessary at this scale.

## Exercises and self-check

1. Derive the bias gradient. **Check:** the average of `p-y`.
2. Why can stronger regularization improve validation while worsening empirical fit? **Check:** it can reduce variance or complexity, but an improvement is not guaranteed.
3. Build a nonlinear synthetic problem and compare raw linear features with a polynomial expansion under the same split. Select the expansion on validation and report the search.
4. A report says “C=10 gave the best test score.” **Check:** test selected C and is no longer an untouched evaluation; use validation or nested evaluation.

## Primary reading

- [scikit-learn: logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression): inspect objective conventions, solver compatibility and penalties.
- [PyTorch: BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.9/generated/torch.nn.BCEWithLogitsLoss.html): compare a stable combined loss with an explicit sigmoid/logarithm calculation.


# PyTorch: autograd and a complete training loop

**Core time:** 2–3 hours. **Prerequisites:** gradients, baselines and Python functions. **Outcomes:** build an MLP; explain every training-step operation; separate training/evaluation modes; save and restore a validation-selected checkpoint.

## 1. Match tensors to the task

Digit inputs are floating tensors of shape `(batch,64)` and targets are integer tensors `(batch,)`. An MLP maps the features through a hidden layer and a nonlinearity to ten raw logits. In this classification mode, cross-entropy expects those logits and integer class indices. Applying softmax first is unnecessary. Floating one-hot targets represent a different supported mode and should not appear accidentally through a dtype conversion.

A parameter is a tensor registered with a module so the optimizer can find it. The forward computation constructs a graph when gradient tracking is enabled. Backward propagates derivatives of the scalar loss through that graph and accumulates them into leaf parameters. The optimizer then reads the gradients and updates parameters. Backward does not itself perform the optimizer step.

## 2. Understand the complete step

Clear previous gradients, compute predictions, compute loss, backpropagate and update. Gradients accumulate by default, so omitting the clearing operation changes the algorithm. Accumulation can be intentional when simulating a larger batch, but losses and partial batches must be scaled consistently. Trace what happens to one weight rather than memorizing an incantation.



In [ ]:
import torch
from torch import nn
torch.manual_seed(7)
model = nn.Sequential(nn.Linear(64,32), nn.ReLU(), nn.Linear(32,10))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
X = torch.rand(12,64)
y = torch.arange(12) % 10
model.train()
optimizer.zero_grad(set_to_none=True)
logits = model(X)
loss = nn.CrossEntropyLoss()(logits,y)
loss.backward()
assert logits.shape == (12,10)
assert all(p.grad is not None and torch.isfinite(p.grad).all()
           for p in model.parameters())
optimizer.step()
print(float(loss.detach()))



The random inputs verify mechanics only. `lab03_neural.py` supplies a real held-out split of the bundled digits and records a validation-selected checkpoint. Check that its parameters change after a step and that the loss remains finite before trusting its accuracy.

## 3. Evaluation mode and gradient mode are independent

`model.eval()` changes dropout and batch-normalization behavior. It does not disable gradient tracking. `torch.no_grad()` avoids building the usual backward graph but does not disable dropout. Use both for ordinary predictive evaluation. During adversarial evaluation, however, keep the network in evaluation mode while enabling input gradients; “evaluation means no gradients” is not a universal rule.

Dropout randomly masks activations during training and uses its defined deterministic inference behavior in evaluation. Batch normalization can maintain running statistics and use them during evaluation. Accidentally evaluating with training behavior may make results depend on batch composition or stochastic masks. These issues can hide behind apparently reasonable aggregate scores.

## 4. Average over the correct denominator

With 130 examples and batch size 64, batch sizes are 64,64,2. The unweighted average of three batch-mean losses gives the final two examples as much influence as each group of 64. Multiply each batch-mean loss by its batch size, sum, then divide by 130. Accuracy counts correct predictions and divides by total examples. Weighted or masked losses may use other denominators; inspect their reduction semantics before reproducing an epoch statistic.

## 5. A checkpoint records a selection decision

Select the checkpoint using a declared validation rule, such as highest validation accuracy and earliest epoch on ties. Deep-copy the best state: a reference to mutable weights can silently become the final epoch instead. Save the state dictionary together with architecture, preprocessing, selected epoch, seed and environment. Resuming training also requires optimizer/scheduler state and relevant random-generator states.

The teaching lab saves only locally created model state. Do not load arbitrary serialized checkpoints from untrusted sources. For your own checkpoint, reload it into the same architecture and compare evaluation logits within a justified tolerance. A seed is not a guarantee of bitwise equality across platforms; record device and deterministic settings.

## 6. Worked diagnosis: loss remains near log(10)

A ten-class classifier with approximately uniform probabilities has loss near `log(10)`. If it stays there, inspect target values and shapes, gradient presence, finite values and parameter changes. Try to overfit 32 examples. If the small-batch test fails, investigate a detached graph, an optimizer linked to a different model instance or labels shuffled separately from inputs. Longer training does not repair those mistakes.

## Guided lab, exercises and self-check

Run `python lab03_neural.py --output outputs/neural`, then inspect history, selected epoch and saved split. Do not tune after reading test output.

1. Call backward twice on two identical forward computations without clearing. **Check:** gradients add; this is not two optimizer updates.
2. Add dropout and compare repeated outputs with `train()` and `eval()`. **Check:** eval removes dropout randomness; `no_grad()` alone does not.
3. Reload the saved best state and report the maximum absolute difference in evaluation logits. Explain the tolerance.
4. Construct uneven batches with very different losses and expose the unweighted-batch-average error. **Check:** use example counts in the denominator.
5. Explain how you would resume a run exactly within one fixed environment. **Check:** weights alone omit optimizer, schedule, random-generator and data-order state.

## Primary reading

- [PyTorch: Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/index.html): connect tensors, data loading, autograd and optimization into one workflow.
- [PyTorch: reproducibility](https://docs.pytorch.org/docs/2.9/notes/randomness.html): document deterministic settings and their limitations.


# Convolutional networks: locality, shared weights and shapes

**Core time:** 120 minutes. **Prerequisites:** affine layers and PyTorch training. **Outcomes:** compute convolution output dimensions and parameter counts; explain weight sharing; implement a small CNN; compare architectures under a declared budget.

## 1. Why preserve image structure?

Flattening an image treats all input positions as separate coordinates. A dense layer can learn useful features but does not explicitly share a detector across positions. A convolutional layer applies the same small filter at many locations. This encodes a preference for local patterns that can appear in multiple places. It reduces parameters relative to a fully connected map of similar spatial size, but does not guarantee better performance for every task.

Deep-learning libraries commonly implement cross-correlation, without reversing the kernel, under the name convolution. A learned filter can still represent the needed patterns; the convention matters when manually reproducing a numerical result. A filter spans all input channels and produces one output channel. Multiple filters create multiple learned feature maps.

## 2. Compute dimensions before running code

For one spatial dimension with input size H, kernel k, padding p, stride s and dilation d, the output size follows the formula below. Apply it independently to height and width. Padding adds boundary values; stride controls sampling locations; dilation spaces filter taps. Their effects are not interchangeable.

```math
H_{out}=\left\lfloor\frac{H+2p-d(k-1)-1}{s}+1\right\rfloor,
\qquad P=C_{out}(C_{in}k_hk_w+1)
```

The parameter formula includes one bias per output channel; remove that term when bias is disabled. With one input channel, eight output channels and 3-by-3 kernels, there are `8*(9+1)=80` parameters. An 8-by-8 input with padding 1 and stride 1 remains 8-by-8. A 2-by-2 pooling operation with stride 2 reduces it to 4-by-4.

## 3. Worked example: one filter response

Take a 3-by-3 patch `[[1,2,3],[4,5,6],[7,8,9]]` and a filter with -1 in its left column, 0 in the middle and 1 in the right. The response is `(-1+3)+(-4+6)+(-7+9)=6`, before bias or activation. This filter responds to a horizontal intensity change. A learned CNN need not learn exactly this filter, but a manual response check confirms indexing conventions.



In [ ]:
import torch
from torch import nn
patch = torch.arange(1.,10.).reshape(1,1,3,3)
kernel = torch.tensor([[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]).reshape(1,1,3,3)
response = torch.nn.functional.conv2d(patch,kernel)
assert response.item() == 6
model = nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),
    nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*4*4,10))
assert model(torch.zeros(5,1,8,8)).shape == (5,10)
print(sum(p.numel() for p in model.parameters()))



The total parameter count is 80 plus `128*10+10`, giving 1,370. Pooling and ReLU have no learned scalar parameters in this architecture. They still perform computation and can affect accuracy, so parameter count alone is not a complete cost measure.

## 4. Receptive fields and invariance need careful language

Stacking local operations grows the region of the input that can influence an output. For stride-one 3-by-3 layers without dilation, two layers have a 5-by-5 theoretical receptive field. Strides and pooling change the spacing between receptive-field centers. The theoretical receptive field only identifies possible influence, not the magnitude of learned influence.

Convolution is translation equivariant under suitable idealized boundary and stride conditions: translating the input translates the feature map. Classification invariance requires additional design or learning and is not exact under every crop, padding rule and downsampling. Do not claim “CNNs are invariant to rotation” merely because convolution shares weights. Data augmentation can encourage particular robustness, but should reflect legitimate label-preserving transformations.

## 5. Compare fairly on tiny images

An 8-by-8 digit has little spatial resolution. Repeated pooling can erase useful structure. A large image architecture copied unchanged can be inappropriate. Compare a compact CNN with a simple MLP using the same split, selection metric and declared training budget. If parameter counts or optimizer schedules differ, report that instead of attributing all differences solely to “convolution.”

Run the CNN option in `lab03_neural.py` as described in the bundle README. Inspect both parameter count and validation trajectory. Treat runtime as a measurement on your CPU, with warm-up and repetitions handled in the efficiency module; one training duration is not an inference benchmark.

## Exercises and self-check

1. For H=15,k=3,p=1,s=2,d=1, find output size. **Check:** 8.
2. A layer has 3 input channels, 16 output channels and 5-by-5 filters with bias. Count parameters. **Check:** `16*(75+1)=1,216`.
3. Why can an unstructured sparse convolution have unchanged dense-kernel latency? **Check:** zeros do not automatically change the operator or hardware workload.
4. Implement a shifted-input test away from boundaries, then show how stride or padding changes the correspondence. Explain which equivariance claim your test actually examines.

## Primary reading

- [PyTorch: Conv2d](https://docs.pytorch.org/docs/2.9/generated/torch.nn.Conv2d.html): verify channel conventions, cross-correlation and output dimensions.
- [Goodfellow, Bengio and Courville: Convolutional Networks](https://www.deeplearningbook.org/contents/convnets.html): study locality and parameter sharing, then connect them to the small example.


# Attention and Transformers: a small calculation before a large model

**Core time:** 2 hours. **Prerequisites:** matrix multiplication, softmax and neural layers. **Outcomes:** track query/key/value shapes; calculate attention weights; explain scaling, masking and positional information; distinguish a toy attention block from a trained language model.

## 1. Attention constructs a data-dependent mixture

In a sequence, a token may need information from other positions. Attention compares a query at one position with keys at candidate positions, normalizes those scores and uses the resulting weights to mix values. Queries and keys determine where to look; values determine what information is combined. They are usually learned projections of the input, not three unrelated external datasets.

For a sequence of length n and hidden width d, input X has shape `(n,d)`. Project to Q and K with key width `d_k`, and V with value width `d_v`. Scores `Q@K.T` have shape `(n,n)`. Row-wise softmax produces one distribution over key positions for each query. Multiplying it by V returns `(n,d_v)`.

```math
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
```

M is an additive mask: permitted entries contribute zero and forbidden entries receive negative infinity in the mathematical idealization. Do not confuse a Boolean mask's library-specific convention with the additive formulation. Different APIs can interpret True differently; read the documentation for the exact function used.

## 2. Worked example with two tokens

Let Q and K both be the 2-by-2 identity and let V contain scalar values `[2,6]`. The scaled score matrix has `1/sqrt(2)` on the diagonal and zero off it. The first row's self-weight is `exp(1/sqrt(2))/(exp(1/sqrt(2))+1)`, approximately 0.6697615. Its output is approximately 3.320954, and the second output is approximately 4.679046. Neither output must equal an original value; attention mixes them.



In [ ]:
import torch
Q = torch.eye(2)
K = torch.eye(2)
V = torch.tensor([[2.],[6.]])
scores = Q@K.T / (Q.shape[-1]**0.5)
weights = scores.softmax(dim=-1)
out = weights@V
torch.testing.assert_close(weights.sum(-1),torch.ones(2))
assert out.shape == (2,1)
mask = torch.triu(torch.ones(2,2,dtype=torch.bool),diagonal=1)
causal = scores.masked_fill(mask,float('-inf')).softmax(-1)@V
torch.testing.assert_close(causal[0],V[0])
print(weights,out,causal)



The first causal position can see only itself, so its output is exactly 2. An all-masked row has no valid probability distribution and can produce NaNs; a valid sequence/mask construction must prevent or explicitly handle that case.

## 3. Why scale and why use multiple heads?

Under a simplified independent zero-mean unit-variance model of query/key coordinates, the variance of their dot product grows with key dimension. Dividing by its square root controls this scale. The assumptions are an intuition, not a guarantee that trained coordinates remain independent. Without reasonable score scales, softmax can become highly concentrated and gradients can become difficult to use.

Multiple heads use separate learned projections and combine the resulting outputs. They allow different learned comparisons, but a head is not automatically assigned a human-interpretable concept. Labeling one “syntax” or “fairness” requires evidence. Attention weights are internal computations and are not a universal causal explanation of a prediction.

## 4. Attention alone does not encode sequence order

Without positional information, self-attention is permutation equivariant under consistent permutation of tokens. A Transformer adds position-related information, residual connections, normalization and feed-forward sublayers. Encoder attention often permits bidirectional context; autoregressive decoder attention uses causal restrictions. Training objectives and data define the task. A matrix-multiplication demonstration is not a pretrained language model and does not show language understanding.

In straightforward full attention, an n-by-n score matrix creates quadratic storage/computation terms in sequence length. Optimized kernels can reduce materialized intermediate storage and improve runtime without changing every aspect of the underlying dense interaction pattern. Claims about memory or asymptotic behavior should name the exact implementation and quantity being measured.

## Guided implementation and exercises

Use the attention checks in `lab03_neural.py`, then:

1. Add a third token, derive the causal mask and verify that changing the future value cannot change the first output. **Check:** with correct masking the first output is independent of future V entries.
2. For batch 4, length 12, heads 3 and key width 8, state per-head Q and score shapes. **Check:** Q `(4,3,12,8)`, scores `(4,3,12,12)` under this layout.
3. Permute a sequence with no positional information and verify consistent output permutation. Add positional vectors and explain why the experiment changes.
4. Compare full and causal attention on the same input. Explain why an autoregressive benchmark would be invalid if future tokens leaked through an incorrect mask.

## Primary reading

- [Vaswani et al.: Attention Is All You Need](https://arxiv.org/abs/1706.03762): read the scaled attention and architecture definitions; distinguish our toy illustration from the paper's trained translation experiments.
- [PyTorch: scaled dot-product attention](https://docs.pytorch.org/docs/2.9/generated/torch.nn.functional.scaled_dot_product_attention.html): inspect mask and dropout semantics before replacing the explicit implementation.


# Evaluation: imbalance, thresholds, calibration and uncertainty

**Core time:** 120 minutes. **Prerequisites:** conditional probability, uncertainty and classification. **Outcomes:** derive metrics from a confusion matrix; distinguish ranking, decisions and probabilities; select thresholds/calibration on validation data; report denominators and uncertainty.

## 1. Start with counts

For a binary task, true positives are positive examples predicted positive; false positives are negative examples predicted positive; false negatives are positive examples predicted negative; true negatives are the remaining correctly rejected negatives. Precision conditions on predicted positives. Recall, also called true-positive rate, conditions on actual positives. The false-positive rate conditions on actual negatives. These denominators must appear in an audit, especially when prevalence is small.

```math
\operatorname{precision}=\frac{TP}{TP+FP},\quad
\operatorname{recall}=\frac{TP}{TP+FN},\quad
F_1=\frac{2TP}{2TP+FP+FN}.
```

A zero denominator means the empirical quantity is undefined under its ordinary definition. A library may substitute zero or emit a warning; disclose the convention. Treating an empty subgroup as a perfect or perfectly fair classifier is particularly misleading.

## 2. Worked example: 95% accuracy can retrieve nothing

In 100 examples with 5 positives, an always-negative classifier has accuracy 95% and positive recall zero. A second classifier finds 4 positives but falsely flags 9 negatives. Its counts are TP=4, FN=1, FP=9, TN=86. Accuracy drops to 90%, precision is 4/13, recall is 4/5 and F1 is 8/18. Which is preferable depends on the consequences and workload, not on one universally best metric.



In [ ]:
tp,fn,fp,tn = 4,1,9,86
metrics = {'accuracy':(tp+tn)/(tp+fn+fp+tn),
    'precision':tp/(tp+fp),'recall':tp/(tp+fn),
    'f1':2*tp/(2*tp+fp+fn)}
assert metrics['accuracy'] == .90
assert metrics['recall'] == .80
print(metrics)



For multiclass problems, macro averaging gives each class equal weight; weighted averaging weights class metrics by support; micro averaging combines counts. These choices answer different questions. Always include class supports and a confusion matrix so the reader can see which classes drive the aggregate.

## 3. Ranking is different from a deployed threshold

ROC-AUC evaluates ranking over true-positive/false-positive rates, while precision–recall summaries emphasize retrieval of positives relative to flagged observations. Their interpretation depends on the task and prevalence. A high AUC does not identify a usable operating point or guarantee calibrated probabilities. Average precision and a trapezoidal area under a precision–recall curve are also not interchangeable definitions; name the metric implementation.

Select thresholds on validation data under a stated objective or constraint, such as maximizing recall subject to a false-positive workload limit. A sample estimate of satisfying that constraint is uncertain. Freeze the threshold before final evaluation. Do not choose a threshold independently for every test subgroup and then present the result as a single preexisting policy.

## 4. Calibration asks whether probabilities mean what they say

Among cases predicted around 0.8 confidence, an appropriately calibrated prediction system should be correct at a compatible frequency on the population under study. Reliability diagrams group predictions into bins and compare confidence with observed accuracy. Bin count, boundaries and sample size affect the picture. Expected calibration error is a useful summary but depends on those choices and can hide local problems.

Negative log-likelihood and Brier score evaluate probabilistic predictions with proper scoring rules under their standard definitions. Multiclass Brier normalization conventions vary, so record yours. Temperature scaling divides logits by one positive scalar fitted on validation data. It preserves argmax labels for a fixed positive temperature while changing confidence, so accuracy can remain unchanged as probabilistic quality changes.

```math
p_k(T)=\frac{\exp(z_k/T)}{\sum_j\exp(z_j/T)},\qquad T>0.
```

Fit T without test labels, then evaluate the frozen transformation on test. Calibration on one distribution does not guarantee calibration under a different distribution. A lower estimated calibration error is not a certificate of safety.

## 5. Evaluation protocol for this program

Report the split and unit of analysis; selection metric; final threshold/calibrator; accuracy, macro F1 and class counts for digits; per-seed scores for stochastic training; and a paired comparison when appropriate. Keep probabilities or logits with example indices so results can be audited. Do not expose personal identifiers in shared artifacts; the bundled data avoid that problem.

Run the evaluation outputs from `lab02_baselines.py` and `lab03_neural.py`. Check that probabilities sum to one and that confusion-matrix counts sum to the test-set size. Compare model predictions on the same indices. If one run used a different split, repair the comparison before computing a paired interval.

## Exercises and self-check

1. Derive specificity and balanced accuracy from the worked counts. **Check:** specificity `86/95`; balanced accuracy is the mean of that value and 0.8.
2. Why can temperature scaling change NLL without changing accuracy? **Check:** a positive common scale preserves the largest logit while changing probability concentration.
3. Design a reliability plot including bin sample counts. Explain how you handle empty bins and why tiny bins are noisy.
4. A paper reports only accuracy on a highly imbalanced dataset. List the additional evidence needed before judging its usefulness. Include operating costs and prevalence as well as more metrics.

## Primary reading

- [scikit-learn: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html): compare definitions and averaging conventions.
- [Guo et al.: On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html): read the temperature-scaling construction and distinguish the published empirical findings from your own experiment.


# Debugging experiments and making results reproducible

**Core time:** 120 minutes. **Prerequisites:** a working training loop and evaluation metrics. **Outcomes:** isolate data/model/optimizer faults; preserve exact selection evidence; distinguish deterministic replay from reproducible conclusions; write an actionable experiment log.

## 1. Debug in order of dependency

Start with data: label range, class counts, input dtype/range, finite values, split overlap and input–target alignment. Then test the forward path: output shape, finite logits and loss, and a hand-checkable example. Next test the backward path: required gradients exist, their shapes match parameters and a step changes weights. Finally test the experimental claim: baseline, budget, selection protocol and correct evaluation population.

A fancy architecture cannot repair mislabeled indices. A deterministic script cannot repair leakage. A passing unit test cannot establish that a benchmark is representative. Each level depends on earlier levels, so debug from simple observable facts toward broad conclusions.

## 2. Tiny-batch overfitting is a diagnostic, not a benchmark

Take a small fixed batch and attempt to reduce its loss substantially with a sufficiently expressive model. Failure suggests a data, objective, optimization or implementation problem. Success shows that the loop can fit that batch; it does not show generalization. Preserve this as a diagnostic separate from the evaluation split, and do not report its accuracy as a model result.



In [ ]:
import torch
from torch import nn
torch.manual_seed(4)
X = torch.randn(16,4)
y = (X[:,0] > 0).long()
net = nn.Sequential(nn.Linear(4,16),nn.Tanh(),nn.Linear(16,2))
opt = torch.optim.Adam(net.parameters(),lr=.03)
initial = float(nn.functional.cross_entropy(net(X),y).detach())
for _ in range(150):
    opt.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(net(X),y)
    loss.backward(); opt.step()
final = float(nn.functional.cross_entropy(net(X),y).detach())
assert final < initial*0.2
print(initial,final)



This deliberately learns a toy batch generated by a known rule. The check tests substantial improvement rather than an exact floating-point score. If it fails in a different environment, inspect the actual trajectory before merely lowering the threshold.

## 3. Negative controls make hidden mistakes visible

A negative control is an experiment expected not to show the desired effect under a justified assumption. Randomizing training labels should make generalization to independent correctly labeled examples poor, although an expressive model can still memorize the randomized training set. If held-out performance remains implausibly high, inspect leakage, duplicated data or whether the evaluation labels were accidentally shared.

Other controls include an untrained model, a majority predictor, disabled augmentation and a zero perturbation in a robustness test. A defense that changes “clean” predictions when its perturbation budget is zero may be evaluating a different preprocessing path. Negative controls need the same logging discipline as the main result.

## 4. Reproducibility has several layers

Exact replay asks whether the same setup produces matching outputs. Computational reproduction asks whether another person can execute the artifacts and obtain compatible results. Scientific robustness asks whether the conclusion survives reasonable changes in seed, split or setting. Passing one layer does not guarantee the others.

Record code revision, Python/dependency versions, device, thread settings, seeds, data checksum, split indices, command, configuration and checkpoint-selection rule. Save raw per-epoch and per-example outputs where appropriate. A chart image alone is insufficient to reconstruct a result. A requirements file helps, but dependencies and platform details can still matter; the bundle records its actual environment on each run.

## 5. Worked example: an apparently better model

Model B achieves higher test accuracy than A, but B was run for 100 epochs while A received 10, and B's best test checkpoint was selected after every epoch. This comparison confounds architecture with training budget and uses test for selection. Repair it by defining a common tuning budget or explicitly comparing complete training procedures, selecting both on validation and assessing the frozen choices on untouched test data. If the original test has already guided development, disclose that and avoid calling it an untouched holdout.

## 6. An issue report a colleague can act on

Provide the smallest command that reproduces the problem, environment, input shapes/ranges, full error, expected behavior and observed behavior. Remove credentials and unrelated personal data. Show the first failing invariant rather than pages of downstream errors. If a run diverges, preserve the configuration and last finite metrics. Editing a result file to match an expected figure is never a fix.

## Exercises and self-check

1. Intentionally omit `zero_grad` and compare gradient norms. **Check:** they accumulate; the effect depends on subsequent parameter changes, so do not expect a simple constant multiplier across a whole training run.
2. Train on randomized labels and describe the distinction between memorization and generalization.
3. Create a reproduction README a peer can follow from an empty output directory. Include runtime expectations as measured on your machine, not universal promises.
4. Classify three failures as numerical, implementation or protocol errors: NaN softmax, a detached loss, and test-selected hyperparameters. **Check:** respectively numerical, implementation and evaluation-protocol errors, though real failures can overlap categories.

## Primary reading

- [PyTorch: reproducibility](https://docs.pytorch.org/docs/2.9/notes/randomness.html): identify controllable and uncontrollable variation.
- [NeurIPS paper checklist](https://nips.cc/public/guides/PaperChecklist): use the evidence questions to improve artifact completeness; completing a checklist is not a claim of publication acceptance.


# ML/DL laboratory: a baseline, an ablation and a defensible conclusion

**Core time:** 3–4 hours. **Prerequisites:** ML/DL 1–7. **Outcomes:** predeclare a comparison; run linear and neural baselines; isolate one design choice; report actual uncertainty and limits; prepare the module assignment.

## 1. Choose a question small enough to answer

A suitable question is: “Under a fixed digit split and validation-selection protocol, does a compact CNN improve predictive performance over a compact MLP at a stated parameter/training budget?” Another is: “How does hidden width affect the validation-selected MLP's accuracy and parameter count?” Do not combine a new architecture, optimizer, augmentation and data split into one change and then attribute the outcome to a single component.

Write the comparison before running it. Specify dataset/source, preprocessing, split indices, models, tuning grid, number of epochs or updates, validation rule, seed schedule, primary metric and secondary metrics. Explain which resource is matched and which is reported. Matching parameter count, training time and update count simultaneously is usually impractical; an honest protocol states the tradeoff.

## 2. The laboratory has a sequence, not a leaderboard

First run `lab02_baselines.py` to establish majority and linear references. Then run `lab03_neural.py` for an MLP and its CNN option using the same split seed. Inspect training/validation histories before inspecting final test outputs. For the assessed extension, use new output directories and a predeclared ablation. Preserve the original baseline artifacts so a later modification cannot silently replace them.

Use at least three training seeds for a stochastic comparison when computationally feasible. Separate the seed controlling the data split from the seed controlling model initialization if you intend to keep the split fixed. Changing both together estimates a different mixture of variation. The bundled core example is intentionally small; your report should identify exactly what varies in your extension.

## 3. Worked protocol and decision rule

Suppose your primary metric is macro F1 and your comparison is hidden width 32 versus 64. Keep preprocessing, split, optimizer family, learning-rate grid, maximum epochs and checkpoint-selection rule fixed. Select each candidate using validation macro F1 or explicitly explain another selection metric. Record all candidate results. Freeze the selected configurations, evaluate test and compute paired differences on matching examples/runs.

Do not announce “width 64 is better” from one decimal place. Report the observed effect, seed variation, finite-sample uncertainty if calculated, and the additional parameter cost. If the difference is uncertain or too small to matter, that is a useful conclusion. An architecture can be a poor practical choice even when its accuracy is slightly higher.

## 4. Evidence table template

| Field | What to record | Why it matters |
|---|---|---|
| Model/config ID | Architecture, width, optimizer and selected hyperparameters | Identifies the evaluated procedure |
| Split hash | Hash of the saved train/validation/test indices | Confirms pairing and prevents accidental split changes |
| Selection | Metric, candidate grid, epoch/tie rule | Makes development decisions auditable |
| Quality | Accuracy, macro F1, class supports and confusion matrix | Reveals failures hidden by an average |
| Variation | Per-seed values and stated uncertainty method | Separates one lucky run from repeatable evidence |
| Cost | Parameters and measured runtime with units/setup | Prevents a one-dimensional comparison |

Do not fill this table with invented sample scores. Generate it from the actual metrics JSON/CSV produced by your runs. Keep fractions and percentages consistent; label accuracy 0.92 or 92%, not “92” without a unit.

## 5. Build the argument from evidence

Begin with the question and scope. State the method sufficiently precisely for a peer to implement it. Present the baseline and ablation results with the same metric definitions. Discuss failure examples and plausible alternative explanations. End with a bounded conclusion: “Under this small dataset and budget, the measured change was…” is stronger scientific communication than an unsupported universal ranking.

Explain limitations such as the low-resolution dataset, one split, CPU-only environment, small seed count or a restricted hyperparameter search. Limitations are not a ceremonial final paragraph; they determine which conclusions are justified. If the ablation failed because of a coding bug, repair and rerun it before interpreting the scientific question.

## Exercises and self-check

1. Write a protocol in which the only experimental factor is dropout probability. Identify which random effects remain even after fixing the intended factor.
2. Design a negative control involving shuffled labels. Predict training and held-out behavior separately.
3. Reproduce the saved checkpoint's predictions in a fresh process, then verify confusion counts and metrics independently.
4. Write two conclusions from the same result: one justified and one exaggerated. Explain precisely which untested population or condition the exaggerated statement adds.

## Submission readiness

Submit code, environment, split identifiers, complete search/selection history, final per-example predictions, plots/tables and a 2–4 page report. Include the command for reproducing the main comparison, a runtime estimate measured on your machine and a short explanation of every assertion. The portal's module assignment contains the scoring rubric. A correct negative result can receive full credit; a high score without a valid protocol cannot.

## Primary reading

- [scikit-learn: common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html): perform a final leakage audit.
- [NeurIPS paper checklist](https://nips.cc/public/guides/PaperChecklist): check whether claims, assumptions, computational requirements and artifacts are supported.
